# Notebook 01 — Data Acquisition: Hourly Electricity Demand of Colombia's SIN

This notebook is the first stage of the MoEForecaster pipeline, a Mixture-of-Experts (MoE) system for forecasting the hourly electricity demand of Colombia's National Interconnected System (SIN).

---

## Objective

Download and validate six calendar years (2020-01-01 to 2025-12-31) of hourly real demand of the SIN from the public SINERGOX API of XM S.A. (the Colombian market operator), and carry out an exploratory data analysis (EDA) of the series: temporal patterns (daily, weekly and annual), autocorrelation structure, data gaps, STL decomposition and stationarity tests. The EDA motivates the modeling choices made in the following notebooks (daily and weekly seasonality, calendar effects, autoregressive lags).

**Place in the pipeline:** 01 data acquisition (this notebook) → 02 preprocessing and feature engineering → 03 clustering of daily and weekly profiles → 04 cluster analysis and cluster classifiers → 05 expert selection → 06 gating design → 07 hyperparameter optimization → 08 production assembly → 09 benchmark.

## Inputs

- **XM S.A. SINERGOX API** (`https://servapibi.xm.com.co/hourly`, metric `DemaReal`, entity `Sistema`): hourly real demand of the SIN, 2020-01-01 to 2025-12-31. No earlier notebook is required.
- If `data/raw/demand.parquet` and `data/raw/metadata.json` already exist (from a previous run of this notebook), they are reused instead of downloading again (set the environment variable `FORCE_DOWNLOAD=1` to force a new download).

## Outputs

| File | Content | Used by |
|---|---|---|
| `data/raw/demand.parquet` | Hourly demand, columns `datetime`, `demand_kwh` (52,608 records for the full period) | Notebook 02 |
| `data/raw/metadata.json` | Dataset metadata (source, period, record count, summary statistics, detected gaps, download timestamp) | Documentation |
| `results/stationarity_results.csv` | ADF and KPSS stationarity test results (Section 6) | Documentation |
| `results/01_*.png` | EDA figures (Section 5) | Documentation |

## Environment setup

This notebook runs unchanged on a local machine (pip/venv or conda), in Docker and in Google Colab.

**`OUTPUT_ROOT`** (next cell) is the folder where the pipeline stores `data/`, `models/` and `results/`:

| Value | Where artifacts are stored |
|---|---|
| `None` (default) | The repository folder, or the folder in the `MOE_OUTPUT_ROOT` environment variable if it is set |
| `"/content/drive/MyDrive/MoEForecaster"` | Google Drive (Colab only; Drive is mounted automatically) |
| `"/any/absolute/path"` or `"relative/path"` | Any folder; relative paths are resolved from the repository root |

Use the **same `OUTPUT_ROOT` in every notebook**: each notebook reads the artifacts written by the previous ones. Setting the `MOE_OUTPUT_ROOT` environment variable once is the easiest way to keep it consistent.

The cell after it finds the repository, which it clones first when running in Colab. In Colab it also installs the pinned dependencies from `requirements.txt` and makes the `moe_forecaster` package in `src/` importable.

In [ ]:
# ============================================================
# PARAMETERS: the only cell you may need to edit
# ============================================================
# Folder for data/, models/ and results/ (see the table above).
# None -> repository folder, or $MOE_OUTPUT_ROOT if that variable is set.
OUTPUT_ROOT = None

In [ ]:
# ============================================================
# ENVIRONMENT BOOTSTRAP: no edits needed
# ============================================================
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/astorress/MoEForecaster.git"
IN_COLAB = "google.colab" in sys.modules


def find_repo_root(start):
    """Walk up from `start` until the folder that contains src/moe_forecaster/."""
    for folder in [start, *start.parents]:
        if (folder / "src" / "moe_forecaster").is_dir():
            return folder
    return None


REPO_ROOT = find_repo_root(Path.cwd().resolve())

if REPO_ROOT is None and IN_COLAB:
    # Colab opens the notebook on its own, without the rest of the repository
    REPO_ROOT = Path("/content/MoEForecaster")
    if not REPO_ROOT.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_ROOT)], check=True)

if REPO_ROOT is None:
    raise RuntimeError(
        "Repository root not found. Start Jupyter from inside the MoEForecaster folder "
        "(local, conda or Docker), or open the notebook in Google Colab."
    )

if IN_COLAB:
    # Install the pinned versions once per Colab runtime. Colab's preinstalled
    # PyTorch (built with CUDA) is kept, so the torch pin is skipped here.
    marker = Path("/content/.moe_requirements_installed")
    if not marker.exists():
        reqs = [
            line for line in (REPO_ROOT / "requirements.txt").read_text().splitlines()
            if line.strip() and not line.startswith(("#", "torch"))
        ]
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *reqs], check=True)
        marker.touch()
        print("Dependencies installed. If Colab asks to restart the session, restart it and run the notebook again.")

if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from moe_forecaster.config import setup_paths

DATA_DIR, MODELS_DIR, RESULTS_DIR = setup_paths(OUTPUT_ROOT, repo_root=REPO_ROOT)

## 1. Configuration

Global constants of the notebook. The output folders (`data/raw`, `results`, ...) are created by the environment setup cell above.

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================
RANDOM_SEED = 42
N_JOBS = -1

import os

## 2. Imports and plotting style

In [ ]:
import json
import time
import warnings
from datetime import datetime, timedelta
from typing import List, Tuple, Optional, Dict, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import seaborn as sns
import holidays
from statsmodels.tsa.seasonal import STL
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

import requests

warnings.filterwarnings("ignore")

# Plot style
sns.set_style("whitegrid")
plt.rcParams.update({
    "figure.figsize": (14, 5),
    "figure.dpi": 120,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "font.family": "serif",
})

print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")

## 3. Data ingestion functions

The public SINERGOX API of XM S.A. returns hourly SIN data through POST requests to:

```
https://servapibi.xm.com.co/hourly
```

**Request used in this notebook:**
- Metric `DemaReal` (real demand of the SIN), entity `Sistema` (whole system), no authentication required.
- The API returns 24 hourly values per day, in kWh; hours are numbered 1–24 and are mapped here to timestamps 00:00–23:00 of the same day.
- One request per **single day** (start date = end date). Multi-day windows were found to return empty or truncated responses, so the download iterates day by day, following the behavior of a reference JavaScript client.

The functions below implement:
1. Generation of one-day query windows covering the requested period.
2. Retries with exponential backoff (3 attempts; 2 s, 4 s, 8 s) on network or HTTP errors.
3. Defensive parsing of the two response formats observed for `HourlyEntities`.
4. Removal of duplicated timestamps and chronological sorting.
5. Validation of the downloaded series: missing values, completeness (24 records per day), plausible range (4,000,000–12,000,000 kWh per hour, i.e. 4–12 GWh) and hourly gaps.

In [ ]:
# ============================================================
# Ingestion functions: XM API (servapibi.xm.com.co)
# Request format aligned with the reference JavaScript client (ReadDB)
# ============================================================
API_URL = "https://servapibi.xm.com.co/hourly"
DEMAND_VARIABLE = "DemaReal"
MAX_RETRIES = 3
INITIAL_BACKOFF = 2  # seconds

# Plausible historical range of hourly SIN demand (kWh): ~4,000-12,000 MWh
DEMAND_MIN_KWH = 4_000_000
DEMAND_MAX_KWH = 12_000_000


def generate_date_windows(
    start_date: str, end_date: str
) -> List[Tuple[str, str]]:
    """
    Generate the date windows used to query the XM API.

    The reference JavaScript client queries hourly data (HourlyEntities)
    with ONE-DAY windows. An earlier version of this function used 30-day
    windows, which probably exceed an internal limit of the API
    (30 x 24 = 720 records per request) and produced empty or truncated
    responses.

    Each window therefore covers exactly one day -> 24 records per request,
    consistent with the behavior of the JavaScript client.

    Parameters
    ----------
    start_date : str  -- Start date 'YYYY-MM-DD'.
    end_date : str    -- End date 'YYYY-MM-DD' (inclusive).

    Returns
    -------
    List[Tuple[str, str]]
        List of (start, end) tuples with start == end (one day per window).
    """
    windows = []
    current = datetime.strptime(start_date, "%Y-%m-%d")
    end = datetime.strptime(end_date, "%Y-%m-%d")

    while current <= end:
        # One-day window: start == end (as in the JavaScript client)
        date_str = current.strftime("%Y-%m-%d")
        windows.append((date_str, date_str))
        current += timedelta(days=1)

    return windows


def _parse_hourly_items(items: list) -> list:
    """
    Parse the XM API response, supporting BOTH observed formats of
    HourlyEntities:

    Format A (list of {Hour, Value}):
        "HourlyEntities": [
            {"Hour": 1, "Value": 5432.1},
            {"Hour": 2, "Value": 5210.3}, ...
        ]

    Format B (list of entities with a Values dict):
        "HourlyEntities": [
            {
                "Id": "Sistema",
                "Values": {"Hour01": 5432.1, "Hour02": 5210.3, ...}
            }
        ]

    This defensive parsing avoids silent failures when the response
    structure differs from the expected one.
    """
    records = []

    for item in items:
        date_str = item.get("Date", "")
        if not date_str:
            continue

        base_date = pd.to_datetime(date_str).normalize()
        hourly_entities = item.get("HourlyEntities", [])

        for entity in hourly_entities:
            # --- Format A: each entity has Hour and Value directly ---
            if "Hour" in entity and "Value" in entity:
                hour = entity["Hour"]  # Hours 1-24 in the API
                value = entity.get("Value")
                if value is not None:
                    dt = base_date + pd.Timedelta(hours=int(hour) - 1)
                    records.append({"datetime": dt, "demand_kwh": float(value)})

            # --- Format B: entity has Id + Values dict ---
            elif "Values" in entity:
                values_dict = entity["Values"]
                for key, value in values_dict.items():
                    # Keys "Hour01", "Hour02", ..., "Hour24"
                    if key.startswith("Hour") and value is not None:
                        try:
                            hour = int(key.replace("Hour", ""))
                            dt = base_date + pd.Timedelta(hours=hour - 1)
                            records.append({"datetime": dt, "demand_kwh": float(value)})
                        except (ValueError, TypeError):
                            continue

            # --- Unknown format: log it for debugging ---
            else:
                print(f"  [WARN] Unexpected structure in HourlyEntities: {list(entity.keys())}")

    return records


def fetch_xm_demand(
    start_date: str,
    end_date: str,
    max_retries: int = MAX_RETRIES,
    initial_backoff: float = INITIAL_BACKOFF,
) -> Optional[pd.DataFrame]:
    """
    Download the hourly real demand of the SIN from the XM API for one
    date window (one day in practice).

    Notes
    -----
    1. The payload includes the field "Filter": [] (present in the
       JavaScript client and missing in an earlier version of this code).
       Some APIs validate the presence of every schema field even if empty.
    2. Defensive parsing supports several response formats.

    Parameters
    ----------
    start_date : str  -- Start date 'YYYY-MM-DD'.
    end_date : str    -- End date 'YYYY-MM-DD'.
    max_retries : int -- Maximum number of attempts.
    initial_backoff : float -- Initial backoff in seconds (doubled after each failed attempt).

    Returns
    -------
    pd.DataFrame or None
        Columns ['datetime', 'demand_kwh'], or None if no data were obtained.
    """
    # "Filter": [] included in the payload, as in the JavaScript client
    payload = {
        "MetricId": DEMAND_VARIABLE,
        "StartDate": start_date,
        "EndDate": end_date,
        "Entity": "Sistema",
        "Filter": [],  # Field present in the JavaScript client
    }
    headers = {"Content-Type": "application/json"}

    for attempt in range(1, max_retries + 1):
        try:
            response = requests.post(
                API_URL, json=payload, headers=headers, timeout=60
            )
            response.raise_for_status()
            data = response.json()

            # Defensive parsing that supports both response formats
            items = data.get("Items", [])
            records = _parse_hourly_items(items)

            if records:
                return pd.DataFrame(records)
            else:
                print(f"  [WARN] No records for {start_date} -> {end_date}")
                return None

        except requests.exceptions.RequestException as e:
            wait = initial_backoff * (2 ** (attempt - 1))
            print(
                f"  [RETRY {attempt}/{max_retries}] Error: {e}. "
                f"Retrying in {wait:.0f}s..."
            )
            time.sleep(wait)

    print(f"  [ERROR] All attempts failed for {start_date} -> {end_date}")
    return None


def download_demand_data(
    start_date: str,
    end_date: str,
) -> pd.DataFrame:
    """
    Download the hourly SIN demand for an arbitrary date range, iterating
    day by day (as in the JavaScript client).

    Note on efficiency: with one-day windows, one year requires ~365
    requests, plus a 0.3 s pause between requests. This is the right
    trade-off: the API answers one-day requests consistently, whereas
    large windows return empty responses.

    Parameters
    ----------
    start_date : str  -- Start date 'YYYY-MM-DD'.
    end_date : str    -- End date 'YYYY-MM-DD' (inclusive).

    Returns
    -------
    pd.DataFrame
        Consolidated DataFrame ['datetime', 'demand_kwh'], sorted, without duplicates.
    """
    windows = generate_date_windows(start_date, end_date)
    print(f"Total windows (days) to query: {len(windows)}")
    print(f"Period: {start_date} -> {end_date}")
    print("=" * 60)

    all_dfs = []
    failed_days = []

    for i, (ws, we) in enumerate(windows, 1):
        #print(f"[{i:3d}/{len(windows)}] Downloading {ws} ...", end=" ")
        df = fetch_xm_demand(ws, we)
        if df is not None:
            all_dfs.append(df)
            #print(f"OK ({len(df)} records)")
        else:
            failed_days.append(ws)
            print("NO DATA")

        # Pause between requests to avoid overloading the API
        # (the JavaScript client uses Promise.all without throttling; being conservative is safer)
        time.sleep(0.3)

    if not all_dfs:
        raise RuntimeError("No data were obtained from the API.")

    # Consolidate
    df_full = pd.concat(all_dfs, ignore_index=True)

    # Remove duplicated timestamps
    n_before = len(df_full)
    df_full = df_full.drop_duplicates(subset="datetime", keep="first")
    n_dupes = n_before - len(df_full)
    if n_dupes > 0:
        print(f"\n[INFO] Removed {n_dupes} duplicated timestamps.")

    # Sort
    df_full = df_full.sort_values("datetime").reset_index(drop=True)

    print(f"\n{'=' * 60}")
    print(f"Download completed: {len(df_full):,} records")
    print(f"Range: {df_full['datetime'].min()} -> {df_full['datetime'].max()}")

    if failed_days:
        print(f"[WARN] Days without data ({len(failed_days)}): {failed_days[:10]}...")

    return df_full


def validate_demand_data(df: pd.DataFrame) -> Dict[str, Any]:
    """
    Validate the completeness and quality of the downloaded demand data.

    Checks:
    1. Missing (null) values.
    2. 24 records per day (detects incomplete days).
    3. Plausible range: DEMAND_MIN_KWH - DEMAND_MAX_KWH (4,000,000 - 12,000,000 kWh).
    4. Temporal gaps (missing hours).

    Parameters
    ----------
    df : pd.DataFrame  -- DataFrame with columns ['datetime', 'demand_kwh'].

    Returns
    -------
    Dict[str, Any]  -- Dictionary with the validation results.
    """
    report = {}

    # --- Null values ---
    n_nulls = df["demand_kwh"].isna().sum()
    report["n_nulls"] = int(n_nulls)
    print(f"Null values in demand_kwh: {n_nulls}")

    # --- Records per day ---
    df_temp = df.copy()
    df_temp["date"] = df_temp["datetime"].dt.date
    records_per_day = df_temp.groupby("date").size()
    incomplete_days = records_per_day[records_per_day != 24]
    report["n_incomplete_days"] = len(incomplete_days)
    if len(incomplete_days) > 0:
        print(f"Days with != 24 records: {len(incomplete_days)}")
        print(incomplete_days.head(10))
    else:
        print("All days have exactly 24 records.")

    # --- Value range ---
    out_of_range = df[
        (df["demand_kwh"] < DEMAND_MIN_KWH) | (df["demand_kwh"] > DEMAND_MAX_KWH)
    ]
    report["n_out_of_range"] = len(out_of_range)
    if len(out_of_range) > 0:
        print(f"\nRecords outside the range [{DEMAND_MIN_KWH}, {DEMAND_MAX_KWH}] kWh: {len(out_of_range)}")
        print(f"  Observed min: {df['demand_kwh'].min():.1f} kWh")
        print(f"  Observed max: {df['demand_kwh'].max():.1f} kWh")
    else:
        print(f"All values within the range [{DEMAND_MIN_KWH}, {DEMAND_MAX_KWH}] kWh.")

    # --- Temporal gaps ---
    expected_range = pd.date_range(
        start=df["datetime"].min(),
        end=df["datetime"].max(),
        freq="h",
    )
    actual_set = set(df["datetime"])
    missing_hours = sorted(set(expected_range) - actual_set)
    report["n_missing_hours"] = len(missing_hours)
    report["missing_hours"] = [str(h) for h in missing_hours[:50]]

    if missing_hours:
        print(f"\nMissing hours (gaps): {len(missing_hours)}")
        gap_blocks = []
        block_start = missing_hours[0]
        prev = missing_hours[0]
        for h in missing_hours[1:]:
            if h - prev > timedelta(hours=1):
                gap_blocks.append((block_start, prev))
                block_start = h
            prev = h
        gap_blocks.append((block_start, prev))
        report["gap_blocks"] = [
            {"start": str(s), "end": str(e)} for s, e in gap_blocks
        ]
        print(f"Contiguous gap blocks: {len(gap_blocks)}")
        for s, e in gap_blocks[:10]:
            duration = (e - s).total_seconds() / 3600 + 1
            print(f"  {s} -> {e}  ({duration:.0f}h)")
    else:
        print("No temporal gaps detected.")
        report["gap_blocks"] = []

    # --- Basic statistics ---
    report["stats"] = {
        "mean": float(df["demand_kwh"].mean()),
        "std": float(df["demand_kwh"].std()),
        "min": float(df["demand_kwh"].min()),
        "max": float(df["demand_kwh"].max()),
        "median": float(df["demand_kwh"].median()),
    }
    print(f"\nBasic statistics:")
    print(f"  Mean:   {report['stats']['mean']:.1f} kWh")
    print(f"  Std:    {report['stats']['std']:.1f} kWh")
    print(f"  Min:    {report['stats']['min']:.1f} kWh")
    print(f"  Max:    {report['stats']['max']:.1f} kWh")
    print(f"  Median: {report['stats']['median']:.1f} kWh")

    return report

## 4. Data download and validation

### 4.1 Download

The full period 2020-01-01 to 2025-12-31 (2,192 days, i.e. 52,608 expected hourly records) is requested. Because the API is queried one day at a time with a 0.3 s pause between requests, a full download takes a long time (on the order of tens of minutes). If `data/raw/demand.parquet` already exists, it is reused together with the download timestamp stored in `data/raw/metadata.json`; set `FORCE_DOWNLOAD=1` to download again.

In [ ]:
# Download the data
START_DATE = "2020-01-01"
END_DATE = "2025-12-31"

# The download makes ~2,200 requests (one per day) and takes ~45 min. If
# data/raw/demand.parquet already exists it is reused; to download again: FORCE_DOWNLOAD=1
raw_path = f"{DATA_DIR}/raw/demand.parquet"
meta_path = f"{DATA_DIR}/raw/metadata.json"
if os.path.exists(raw_path) and os.environ.get("FORCE_DOWNLOAD") != "1":
    df = pd.read_parquet(raw_path)
    with open(meta_path) as f:
        DOWNLOAD_TIMESTAMP = json.load(f)["download_timestamp"]
    print(f"Using previously downloaded data: {raw_path} ({len(df):,} records, downloaded on {DOWNLOAD_TIMESTAMP})")
else:
    df = download_demand_data(START_DATE, END_DATE)
    DOWNLOAD_TIMESTAMP = datetime.now().isoformat()

### 4.2 Validation

The downloaded series is checked for null values, days with a number of records other than 24, values outside the plausible range and missing hours. The validation report is stored in the metadata file (Section 7). The validation only reports; it does not modify or remove any record.

In [ ]:
# Validate
print("\n" + "=" * 60)
print("DATA VALIDATION")
print("=" * 60)
validation_report = validate_demand_data(df)

print(f"\nFinal shape: {df.shape}")
df.head(10)

---

## 5. Exploratory data analysis (EDA)

This section characterizes the demand series in order to support the modeling decisions of the pipeline: the daily, weekly and annual patterns, the autocorrelation structure, the presence of gaps and the trend/seasonal components. These analyses motivate, in particular, the clustering of daily and weekly profiles (notebook 03) and the calendar, Fourier and lag features (notebook 02).

### 5.1 Full time series

Hourly demand over the whole period, with a centered 7-day (168 h) moving average to highlight the trend.

In [ ]:
# --- 5.1 Full time series ---
fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(df["datetime"], df["demand_kwh"], linewidth=0.3, alpha=0.8, color="#2c3e50")

# Weekly moving average to show the trend
df_plot = df.set_index("datetime").copy()
rolling_mean = df_plot["demand_kwh"].rolling(window=168, center=True).mean()
ax.plot(rolling_mean.index, rolling_mean.values, linewidth=1.5, color="#e74c3c",
        label="7-day moving average")

ax.set_title("Hourly Electricity Demand of the Colombian SIN", fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Demand (kWh)")
ax.legend(loc="upper right")
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_minor_locator(mdates.MonthLocator(bymonth=[4, 7, 10]))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/01_timeseries_full.png", dpi=150, bbox_inches="tight")
plt.show()

### 5.2 Demand distribution by hour of the day

The hourly load profile of the SIN reflects the cycle of economic and residential activity: minimum demand in the early morning (around 03:00–04:00), a morning ramp, a midday peak, a slight early-afternoon dip and an evening peak (around 19:00–20:00). This intra-day pattern motivates the daily-profile clustering and the hour-of-day features.

In [ ]:
# --- 5.2 Distribution by hour of the day ---
df["hour"] = df["datetime"].dt.hour

fig, ax = plt.subplots(figsize=(14, 6))
bp = ax.boxplot(
    [df[df["hour"] == h]["demand_kwh"].dropna().values for h in range(24)],
    positions=range(24),
    widths=0.6,
    patch_artist=True,
    showfliers=True,
    flierprops={"marker": ".", "markersize": 2, "alpha": 0.3},
    medianprops={"color": "#c0392b", "linewidth": 1.5},
)

# Color boxes with a gradient
colors = plt.cm.YlOrRd(np.linspace(0.15, 0.85, 24))
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_title("Demand Distribution by Hour of the Day", fontweight="bold")
ax.set_xlabel("Hour of the day")
ax.set_ylabel("Demand (kWh)")
ax.set_xticks(range(24))
ax.set_xticklabels([f"{h:02d}" for h in range(24)])
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/01_dist_by_hour.png", dpi=150, bbox_inches="tight")
plt.show()

### 5.3 Demand distribution by day of the week

Demand shows a marked weekly pattern: working days (Monday to Friday) have higher demand than weekends (Saturday and Sunday), with Sundays showing the lowest levels. Holidays, which also depress demand, are not separated in this plot; they are treated explicitly through the holiday features of notebook 02.

In [ ]:
# --- 5.3 Distribution by day of the week ---
df["day_of_week"] = df["datetime"].dt.dayofweek  # 0=Monday, 6=Sunday
day_labels = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

fig, ax = plt.subplots(figsize=(10, 6))
bp = ax.boxplot(
    [df[df["day_of_week"] == d]["demand_kwh"].dropna().values for d in range(7)],
    positions=range(7),
    widths=0.6,
    patch_artist=True,
    showfliers=True,
    flierprops={"marker": ".", "markersize": 2, "alpha": 0.3},
    medianprops={"color": "#c0392b", "linewidth": 1.5},
)

weekday_colors = ["#3498db"] * 5 + ["#e67e22"] * 2
for patch, color in zip(bp["boxes"], weekday_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_title("Demand Distribution by Day of the Week", fontweight="bold")
ax.set_xlabel("Day of the week")
ax.set_ylabel("Demand (kWh)")
ax.set_xticks(range(7))
ax.set_xticklabels(day_labels)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/01_dist_by_day_of_week.png", dpi=150, bbox_inches="tight")
plt.show()

### 5.4 Demand distribution by month

Colombia lies in the equatorial zone and has no marked thermal seasons. Electricity demand nevertheless varies along the year due to factors such as holiday seasons (December–January, Holy Week), industrial activity and climate phenomena (El Niño / La Niña, which affect hydroelectric generation and, indirectly, demand).

In [ ]:
# --- 5.4 Distribution by month ---
df["month"] = df["datetime"].dt.month
month_labels = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

fig, ax = plt.subplots(figsize=(14, 6))
bp = ax.boxplot(
    [df[df["month"] == m]["demand_kwh"].dropna().values for m in range(1, 13)],
    positions=range(12),
    widths=0.6,
    patch_artist=True,
    showfliers=True,
    flierprops={"marker": ".", "markersize": 2, "alpha": 0.3},
    medianprops={"color": "#c0392b", "linewidth": 1.5},
)

month_colors = plt.cm.viridis(np.linspace(0.1, 0.9, 12))
for patch, color in zip(bp["boxes"], month_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_title("Demand Distribution by Month", fontweight="bold")
ax.set_xlabel("Month")
ax.set_ylabel("Demand (kWh)")
ax.set_xticks(range(12))
ax.set_xticklabels(month_labels)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/01_dist_by_month.png", dpi=150, bbox_inches="tight")
plt.show()

### 5.5 Heatmap: hour of the day × day of the week

Mean demand for every combination of hour of the day and day of the week. It shows the daily and weekly patterns simultaneously and identifies the hours and days of highest and lowest demand.

In [ ]:
# --- 5.5 Heatmap hour x day of the week ---
pivot = df.pivot_table(
    values="demand_kwh",
    index="hour",
    columns="day_of_week",
    aggfunc="mean",
)
pivot.columns = day_labels

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    pivot,
    cmap="YlOrRd",
    annot=True,
    fmt=".0f",
    linewidths=0.5,
    linecolor="white",
    cbar_kws={"label": "Mean demand (kWh)"},
    ax=ax,
)
ax.set_title("Mean Demand (kWh): Hour of the Day vs Day of the Week", fontweight="bold")
ax.set_xlabel("Day of the week")
ax.set_ylabel("Hour of the day")
ax.set_yticklabels([f"{h:02d}:00" for h in range(24)], rotation=0)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/01_heatmap_hour_day.png", dpi=150, bbox_inches="tight")
plt.show()

### 5.6 Autocorrelation (ACF / PACF)

The autocorrelation and partial autocorrelation functions are computed up to lag 336 (14 days = 2 weeks), with 95% confidence bands, to identify:
- **Lag 24:** daily periodicity (the hourly pattern repeats every day).
- **Lag 168:** weekly periodicity (the pattern repeats every week).
- **Lag 336:** two weeks, to check the persistence of the weekly cycle.

These periodicities motivate the lag set of notebook 02 (lags around 24, 48 and 168 h).

In [ ]:
# --- 5.6 ACF / PACF ---
series = df.set_index("datetime")["demand_kwh"].dropna()

fig, axes = plt.subplots(2, 1, figsize=(16, 8))

plot_acf(series, lags=336, ax=axes[0], alpha=0.05,
         title="Autocorrelation (ACF) — Hourly Demand")
axes[0].axvline(x=24, color="#e74c3c", linestyle="--", alpha=0.7, label="Lag 24 (daily)")
axes[0].axvline(x=168, color="#2ecc71", linestyle="--", alpha=0.7, label="Lag 168 (weekly)")
axes[0].legend(loc="upper right")

plot_pacf(series, lags=336, ax=axes[1], alpha=0.05, method="ywm",
          title="Partial Autocorrelation (PACF) — Hourly Demand")
axes[1].axvline(x=24, color="#e74c3c", linestyle="--", alpha=0.7, label="Lag 24 (daily)")
axes[1].axvline(x=168, color="#2ecc71", linestyle="--", alpha=0.7, label="Lag 168 (weekly)")
axes[1].legend(loc="upper right")

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/01_acf_pacf.png", dpi=150, bbox_inches="tight")
plt.show()

### 5.7 Detection of temporal gaps

The continuity of the series is checked by comparing the observed timestamps with a complete hourly range between the first and last timestamps. Gaps can be caused by API unavailability, system maintenance or missing reports. If gaps exist, the number of missing hours per day is plotted.

In [ ]:
# --- 5.7 Gap detection ---
full_range = pd.date_range(
    start=df["datetime"].min(),
    end=df["datetime"].max(),
    freq="h",
)
actual_timestamps = set(df["datetime"])
missing = sorted(set(full_range) - actual_timestamps)

print(f"Expected range: {full_range[0]} -> {full_range[-1]}")
print(f"Total expected hours: {len(full_range):,}")
print(f"Total hours present:  {len(actual_timestamps):,}")
print(f"Missing hours:        {len(missing):,}")

if missing:
    # Plot the gaps
    gap_df = pd.DataFrame({"datetime": missing})
    gap_df["date"] = gap_df["datetime"].dt.date
    gap_counts = gap_df.groupby("date").size().reset_index(name="missing_hours")

    fig, ax = plt.subplots(figsize=(14, 4))
    ax.bar(
        pd.to_datetime(gap_counts["date"]),
        gap_counts["missing_hours"],
        color="#e74c3c",
        width=1,
    )
    ax.set_title("Missing Hours per Day (Temporal Gaps)", fontweight="bold")
    ax.set_xlabel("Date")
    ax.set_ylabel("Missing hours")
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    plt.tight_layout()
    plt.savefig(f"{RESULTS_DIR}/01_temporal_gaps.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("\nNo temporal gaps detected. The series is complete.")

### 5.8 Descriptive statistics by period

Descriptive statistics of hourly demand grouped by year, by month (pooled over all years) and by hour of the day, to quantify the variability of demand at different temporal granularities. The monthly mean demand is also plotted to show the long-term trend.

In [ ]:
# --- 5.8 Descriptive statistics by period ---
df["year"] = df["datetime"].dt.year

# By year
print("=" * 60)
print("STATISTICS BY YEAR")
print("=" * 60)
yearly_stats = df.groupby("year")["demand_kwh"].agg(
    ["count", "mean", "std", "min", "median", "max"]
).round(1)
yearly_stats.columns = ["N", "Mean", "Std", "Min", "Median", "Max"]
print(yearly_stats.to_string())

# By month (pooled over all years)
print(f"\n{'=' * 60}")
print("STATISTICS BY MONTH (pooled over all years)")
print("=" * 60)
monthly_stats = df.groupby("month")["demand_kwh"].agg(
    ["mean", "std", "min", "max"]
).round(1)
monthly_stats.index = month_labels
monthly_stats.columns = ["Mean", "Std", "Min", "Max"]
print(monthly_stats.to_string())

# By hour
print(f"\n{'=' * 60}")
print("STATISTICS BY HOUR OF THE DAY")
print("=" * 60)
hourly_stats = df.groupby("hour")["demand_kwh"].agg(
    ["mean", "std", "min", "max"]
).round(1)
hourly_stats.index = [f"{h:02d}:00" for h in range(24)]
hourly_stats.columns = ["Mean", "Std", "Min", "Max"]
print(hourly_stats.to_string())

# Plot the monthly mean demand over time
fig, ax = plt.subplots(figsize=(12, 5))
monthly_mean = df.groupby(["year", "month"])["demand_kwh"].mean().reset_index()
monthly_mean["date"] = pd.to_datetime(
    monthly_mean["year"].astype(str) + "-" + monthly_mean["month"].astype(str) + "-15"
)
ax.plot(monthly_mean["date"], monthly_mean["demand_kwh"], "o-",
        color="#2c3e50", markersize=4, linewidth=1.2)
ax.set_title("Monthly Mean Demand", fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Mean demand (kWh)")
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_minor_locator(mdates.MonthLocator(bymonth=[4, 7, 10]))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/01_monthly_demand.png", dpi=150, bbox_inches="tight")
plt.show()

### 5.9 STL decomposition (Seasonal-Trend decomposition using LOESS)

Robust STL decomposition is applied with two seasonal periods, separately:
- **P = 24:** daily component (hourly pattern within each day).
- **P = 168:** weekly component (pattern that repeats every week).

STL separates the series into trend, seasonal and residual components, which allows a visual check of the seasonality and trend before modeling. If the series has gaps, they are filled by linear interpolation **only for this decomposition** (STL requires a regular, complete series); the saved data are not interpolated.

Each decomposition is shown for the full series and, for readability, for a single month (January 2022), where the intra-day and intra-week structure is visible.

In [ ]:
# --- 5.9 STL decomposition ---
# Build a series with a regular hourly index
ts = df.set_index("datetime")["demand_kwh"].copy()
ts = ts.asfreq("h")

# Fill NaNs (if there are gaps) by linear interpolation so that STL can run
n_nans = ts.isna().sum()
if n_nans > 0:
    print(f"Filling {n_nans} NaN values by linear interpolation for STL...")
    ts = ts.interpolate(method="linear")

# --- STL with daily period (P=24), full series ---
print("Running STL with daily period (P=24)...")
stl_daily = STL(ts, period=24, robust=True)
result_daily = stl_daily.fit()

fig, axes = plt.subplots(4, 1, figsize=(16, 12), sharex=True)
components = [
    ("Original Series", ts, "#2c3e50"),
    ("Trend", result_daily.trend, "#e74c3c"),
    ("Daily Seasonality (P=24)", result_daily.seasonal, "#3498db"),
    ("Residual", result_daily.resid, "#95a5a6"),
]
for ax, (title, data, color) in zip(axes, components):
    ax.plot(data.index, data.values, linewidth=0.3, color=color, alpha=0.8)
    ax.set_ylabel("kWh")
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

axes[-1].set_xlabel("Date")
axes[-1].xaxis.set_major_locator(mdates.YearLocator())
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
fig.suptitle("STL Decomposition — Daily Period (P=24)", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/01_stl_daily.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# --- STL with daily period (P=24), January 2022 only ---
df_2022_jan = df[(df['datetime'].dt.year == 2022) & (df['datetime'].dt.month == 1)].copy()

# Build a series with a regular hourly index
ts_jan2022 = df_2022_jan.set_index("datetime")["demand_kwh"].copy()
ts_jan2022 = ts_jan2022.asfreq("h")

# Fill NaNs (if there are gaps) by linear interpolation so that STL can run
n_nans = ts_jan2022.isna().sum()
if n_nans > 0:
    print(f"Filling {n_nans} NaN values by linear interpolation for STL...")
    ts_jan2022 = ts_jan2022.interpolate(method="linear")

# NOTE: this overwrites `result_daily` (full series) with the January 2022 fit;
# Section 6 uses `result_daily` as it stands after this cell.
print("Running STL with daily period (P=24)...")
stl_daily = STL(ts_jan2022, period=24, robust=True)
result_daily = stl_daily.fit()

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
components = [
    ("Original Series", ts_jan2022, "#0052CC"),   # strong blue
    ("Trend", result_daily.trend, "#D7263D"),     # intense red
    ("Daily Seasonality (P=24)", result_daily.seasonal, "#00A6D6"),
    ("Residual", result_daily.resid, "#7B2CBF"),  # strong purple
]

for ax, (title, data, color) in zip(axes, components):
    ax.plot(data.index, data.values, linewidth=0.6, color=color, alpha=0.8)
    ax.set_ylabel("kWh")
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

axes[-1].set_xlabel("Date")
axes[-1].xaxis.set_major_locator(mdates.WeekdayLocator())  # one tick per week
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%D"))
fig.suptitle("STL Decomposition — Daily Period (P=24), January 2022", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/01_stl_daily_jan2022.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# --- STL with weekly period (P=168), full series ---
print("Running STL with weekly period (P=168)...")
stl_weekly = STL(ts, period=168, robust=True)
result_weekly = stl_weekly.fit()

fig, axes = plt.subplots(4, 1, figsize=(16, 12), sharex=True)
components = [
    ("Original Series", ts, "#2c3e50"),
    ("Trend", result_weekly.trend, "#e74c3c"),
    ("Weekly Seasonality (P=168)", result_weekly.seasonal, "#27ae60"),
    ("Residual", result_weekly.resid, "#95a5a6"),
]
for ax, (title, data, color) in zip(axes, components):
    ax.plot(data.index, data.values, linewidth=0.3, color=color, alpha=0.8)
    ax.set_ylabel("kWh")
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

axes[-1].set_xlabel("Date")
axes[-1].xaxis.set_major_locator(mdates.YearLocator())
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
fig.suptitle("STL Decomposition — Weekly Period (P=168)", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/01_stl_weekly.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# --- STL with weekly period (P=168), January 2022 only ---
print("Running STL with weekly period (P=168)...")
stl_weekly = STL(ts_jan2022, period=168, robust=True)
result_weekly = stl_weekly.fit()

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
components = [
    ("Original Series", ts_jan2022, "#0052CC"),    # strong blue
    ("Trend", result_weekly.trend, "#D7263D"),     # intense red
    ("Weekly Seasonality (P=168)", result_weekly.seasonal, "#00A6D6"),
    ("Residual", result_weekly.resid, "#7B2CBF"),  # strong purple
]

for ax, (title, data, color) in zip(axes, components):
    ax.plot(data.index, data.values, linewidth=0.6, color=color, alpha=0.8)
    ax.set_ylabel("kWh")
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

axes[-1].set_xlabel("Date")
axes[-1].xaxis.set_major_locator(mdates.WeekdayLocator())  # one tick per week
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%D"))
fig.suptitle("STL Decomposition — Weekly Period (P=168), January 2022", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/01_stl_weekly_jan2022.png", dpi=150, bbox_inches="tight")
plt.show()

---

## 6. Stationarity analysis

Two complementary unit-root/stationarity tests are applied at the 5% significance level:

| Test | H₀ | Decision rule used in the code |
|---|---|---|
| **ADF** (Augmented Dickey-Fuller, lag order selected by AIC) | The series has a unit root (non-stationary) | Stationary if p ≤ 0.05 and statistic < 5% critical value |
| **KPSS** (Kwiatkowski-Phillips-Schmidt-Shin, level stationarity, `regression='c'`) | The series is level-stationary | Non-stationary if p ≤ 0.05 and statistic > 5% critical value |

The tests are applied to (i) the hourly demand series `ts` (Section 5.9, gaps filled by linear interpolation) and (ii) the residual component of the daily STL decomposition stored in `result_daily`.

**Note:** when the notebook is run from top to bottom, `result_daily` holds the STL fit of the **January 2022** subset (the cell that produces `01_stl_daily_jan2022.png` reassigns it), so test (ii) is computed on the residuals of that month, not of the full series. The results are written to `results/stationarity_results.csv`.

In [ ]:
from statsmodels.tsa.stattools import adfuller, kpss

def run_stationarity_tests(series, series_name):
    """Run the ADF and KPSS tests on a series and return the formatted results."""
    # NOTE: the dictionary keys below are the column names of the saved
    # results/stationarity_results.csv and are kept unchanged.
    results = []

    # --- ADF test ---
    adf_result = adfuller(series, autolag='AIC')
    adf_statistic = adf_result[0]
    adf_pvalue = adf_result[1]
    adf_critical_value = adf_result[4]['5%']
    adf_conclusion = "Stationary (rejects H0)" if adf_pvalue <= 0.05 and adf_statistic < adf_critical_value else "Non-stationary (does not reject H0)"
    results.append({
        "Series": series_name,
        "Test": "ADF (Augmented Dickey-Fuller)",
        "Statistic": round(adf_statistic, 4),
        "p-value": round(adf_pvalue, 4),
        "Critical value (5%)": adf_critical_value,
        "Conclusion": adf_conclusion,
    })

    # --- KPSS test ---
    kpss_result = kpss(series, regression='c', nlags='auto')  # 'c' for level stationarity
    kpss_statistic = kpss_result[0]
    kpss_pvalue = kpss_result[1]
    kpss_critical_value = kpss_result[3]['5%']
    kpss_conclusion = "Non-stationary (rejects H0)" if kpss_pvalue <= 0.05 and kpss_statistic > kpss_critical_value else "Stationary (does not reject H0)"
    results.append({
        "Series": series_name,
        "Test": "KPSS (Kwiatkowski-Phillips-Schmidt-Shin)",
        "Statistic": round(kpss_statistic, 4),
        "p-value": round(kpss_pvalue, 4),
        "Critical value (5%)": kpss_critical_value,
        "Conclusion": kpss_conclusion,
    })

    return results

print("=" * 60)
print("STATIONARITY ANALYSIS")
print("=" * 60)

# --- Original series for the analysis ---
# 'ts' should already be defined by the STL decomposition cells (Section 5.9).
# If not, recreate it:
if 'ts' not in locals():
    ts = df.set_index("datetime")["demand_kwh"].copy()
    ts = ts.asfreq("h")
    if ts.isna().sum() > 0:
        ts = ts.interpolate(method="linear")

# --- STL residuals (daily STL decomposition) ---
# 'result_daily' should be available from the STL decomposition cells (Section 5.9).
# If not, run the daily STL decomposition to obtain the residuals.
if 'result_daily' not in locals():
    print("[WARN] result_daily not found. Running daily STL for residuals...")
    stl_daily = STL(ts, period=24, robust=True)
    result_daily = stl_daily.fit()

# Run the tests
stationarity_results = []
stationarity_results.extend(run_stationarity_tests(ts, "Original series"))
stationarity_results.extend(run_stationarity_tests(result_daily.resid.dropna(), "STL residuals (daily)"))

# Show the results as a DataFrame
stationarity_df = pd.DataFrame(stationarity_results)
print(stationarity_df.to_string(index=False))

In [ ]:
stationarity_df.to_csv(f"{RESULTS_DIR}/stationarity_results.csv", index=False)

---

## 7. Saving the data

The downloaded series (columns `datetime`, `demand_kwh`, sorted chronologically, without the auxiliary columns added during the EDA) is saved in **Parquet** format (compact and fast to read) to `data/raw/demand.parquet`, which is the input of notebook 02. A **JSON metadata** file (`data/raw/metadata.json`) documents the source, period, number of records and days, summary statistics, the gaps detected by the validation of Section 4.2, and the download timestamp.

In [ ]:
# --- 7. Save data ---

# Prepare a clean DataFrame to save
df_save = df[["datetime", "demand_kwh"]].copy()
df_save = df_save.sort_values("datetime").reset_index(drop=True)

# Save Parquet
parquet_path = f"{DATA_DIR}/raw/demand.parquet"
df_save.to_parquet(parquet_path, engine="pyarrow", index=False)
parquet_size_mb = os.path.getsize(parquet_path) / (1024 * 1024)
print(f"Parquet saved: {parquet_path} ({parquet_size_mb:.2f} MB)")

# Build metadata
metadata = {
    "project": "MoE Electricity Demand Forecasting - SIN Colombia",
    "notebook": "01_data_acquisition",
    "variable": "DemaReal (real demand of the SIN)",
    "source": "XM S.A. E.S.P. - API SINERGOX",
    "api_url": API_URL,
    "resolution": "hourly",
    "unit": "kWh",
    "start_date": str(df_save["datetime"].min()),
    "end_date": str(df_save["datetime"].max()),
    "n_records": len(df_save),
    "n_days": int(df_save["datetime"].dt.date.nunique()),
    "n_nulls": int(df_save["demand_kwh"].isna().sum()),
    "demand_stats": {
        "mean_kwh": round(float(df_save["demand_kwh"].mean()), 2),
        "std_kwh": round(float(df_save["demand_kwh"].std()), 2),
        "min_kwh": round(float(df_save["demand_kwh"].min()), 2),
        "max_kwh": round(float(df_save["demand_kwh"].max()), 2),
    },
    "detected_gaps": validation_report.get("gap_blocks", []),
    "n_missing_hours": validation_report.get("n_missing_hours", 0),
    "download_timestamp": DOWNLOAD_TIMESTAMP,
    "parquet_file": "demand.parquet",
    "parquet_size_mb": round(parquet_size_mb, 2),
}

# Save JSON metadata
metadata_path = f"{DATA_DIR}/raw/metadata.json"
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False, default=str)

print(f"Metadata saved: {metadata_path}")
print(f"\nSummary:")
print(f"  Records: {metadata['n_records']:,}")
print(f"  Days: {metadata['n_days']:,}")
print(f"  Period: {metadata['start_date']} -> {metadata['end_date']}")
print(f"  Detected gaps: {metadata['n_missing_hours']} hours")
print(f"  Parquet size: {metadata['parquet_size_mb']} MB")